# Cell segmentation using Stardist

## Import packages

We import the necessary packages. The `ISS_postprocessing` module uses **StarDist**, which can take advantage of a **CUDA-compatible GPU** for faster segmentation.

A GPU is **recommended**, especially for large images, but the code will still run on **CPU** if no GPU is available (with slower performance).

In [ ]:
import ISS_postprocessing.segmentation as SEG

## Cell segmentation using StarDist

In this step we build a **segmentation mask** from the DAPI signal using **StarDist**.

The segmentation function supports two input modes:

**1. Retiled mode** (`input_image_type="retiled"`)

All DAPI tiles in

```
/preprocessing/Cycle1/4_retiled/
```

(e.g. `Cycle1_s0_ch4.tif`, `Cycle1_s1_ch4.tif`, …) are segmented individually.  
Tile positions from

```
/preprocessing/Cycle1/4_retiled/Cycle1_retiled_coords.csv
```

are then used to stitch the tiles into one full segmentation mask:

```
{region}_stardist_retiled_expanded.npz
```

**2. Stitched mode** (`input_image_type="stitched"`)

StarDist runs directly on the stitched image:

```
/preprocessing/Cycle1/3_stitched/Cycle1_ch4.tif
```

The result is saved as:

```
{region}_stardist_stitched_expanded.npz
```

**Retiled mode is recommended for very large images that may exceed GPU memory when processed as a single stitched image.**

### Function parameters

`input_dir` (str)  
Path to the parent directory containing the preprocessed region folders (for example `/R1/`, `/R2/`, …).

`region` (str)  
Identifier of the region to process.

`output_dir_prefix` (str or None)  
Optional alternative output root. If provided, segmentation outputs are written there instead of inside the region folder.

`DAPI_ch` (int)  
Channel number used for the DAPI image. Default is `4`.

`input_image_type` (`"retiled"` or `"stitched"`)  
Selects whether StarDist runs on retiled images or directly on the stitched image.

`model_name` (str)  
Name of the pretrained StarDist model. Default is `"2D_versatile_fluo"`.

`expand_cells` (bool)  
If `True`, expand labels after segmentation.

`n_tiles` (tuple of int)  
Tiling used internally by StarDist during prediction.

`expanded_distance` (int)  
Number of pixels used when expanding labels.

`auto_select_gpu` (bool)  
If `True`, automatically selects a relatively free GPU before running segmentation.

> **Note:**  
> You need to specify the names of the regions you want to post-process.  
> Multiple regions can be processed in one run, and they will all be analyzed using the same parameters.  
> In most cases this is fine, but there may be situations where individual regions need to be treated differently.  


In [ ]:
input_dir = '/path/to/regions/'
regions = ['R1', 'R2', 'R3']

In [ ]:
for region in regions:

    SEG.stardist_segmentation(
        input_dir,
        region,
        DAPI_ch=4,
        output_dir_prefix = None,    # or '/path/to/preferred/output/dir'
        input_image_type="retiled",  # "retiled" or "stitched"
        model_name="2D_versatile_fluo",
        expand_cells=True,
        n_tiles=(4, 4),
        expanded_distance=20,
        auto_select_gpu=True,
    )